# P2: Distributed Training at Scale

**Module 3 — Week 11**

## Problem Statement
Train ML models at scale using SageMaker-compatible patterns. Demonstrate that the same training script works locally and (optionally) in the cloud.

## Metrics
- **Primary:** ROC-AUC on hold-out test set
- **Secondary:** F1-score, Precision, Recall
- **Operational:** Training time, artifact size

---
## 1. Setup

In [ ]:
import sys, os, json, time, tempfile, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, RocCurveDisplay
)

# Add src to path
sys.path.insert(0, str(Path.cwd()))
from src.generate_large_data import generate_large_dataset

sns.set_style("whitegrid")
%matplotlib inline
print("Setup complete.")

In [ ]:
# Generate or load dataset
data_path = Path("data/large_churn_data.parquet")

if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f"Loaded from {data_path}")
else:
    df = generate_large_dataset()
    data_path.parent.mkdir(exist_ok=True)
    df.to_parquet(data_path, index=False)
    print(f"Generated and saved to {data_path}")

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['churned'].mean():.1%}")
df.head()

In [ ]:
# Basic EDA
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum().sum())
print("\nDescriptive stats (numeric):")
df.describe()

---
## 2. Data Preparation

Split into train/val/test and save to a SageMaker-like channel directory structure.

In [ ]:
# Encode categorical features
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

df_encoded = df.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

# Features and target
feature_cols = [c for c in df_encoded.columns if c != "churned"]
X = df_encoded[feature_cols].values
y = df_encoded["churned"].values

# 70 / 15 / 15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")

In [ ]:
# Save to SageMaker-like channel structure
channel_dir = Path("data/channels")
for split_name, X_split, y_split in [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
]:
    split_dir = channel_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    # SageMaker convention: target in first column for CSV
    combined = np.column_stack([y_split, X_split])
    np.savetxt(split_dir / "data.csv", combined, delimiter=",", fmt="%.6f")

# Save feature names for later
with open(channel_dir / "feature_names.json", "w") as f:
    json.dump(feature_cols, f)

print(f"Channels saved to {channel_dir}")
print("SM_CHANNEL_TRAIN =", str(channel_dir / "train"))
print("SM_CHANNEL_VALIDATION =", str(channel_dir / "validation"))

---
## 3. SageMaker-Compatible Training Script

Write a training script that reads data from environment-variable paths (just like SageMaker Script Mode).

In [ ]:
training_script = '''
#!/usr/bin/env python3
"""SageMaker-compatible training script for churn prediction.

Reads data from SM_CHANNEL_TRAIN / SM_CHANNEL_VALIDATION.
Saves model to SM_MODEL_DIR.
"""
import os, json, pickle, argparse
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score

def load_channel(channel_dir):
    """Load CSV from a SageMaker channel directory."""
    data = np.loadtxt(os.path.join(channel_dir, "data.csv"), delimiter=",")
    y = data[:, 0].astype(int)
    X = data[:, 1:]
    return X, y

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=5)
    parser.add_argument("--learning-rate", type=float, default=0.1)
    args = parser.parse_args()

    # SageMaker env vars (with local defaults)
    train_dir = os.environ.get("SM_CHANNEL_TRAIN", "data/channels/train")
    val_dir = os.environ.get("SM_CHANNEL_VALIDATION", "data/channels/validation")
    model_dir = os.environ.get("SM_MODEL_DIR", "outputs/model")
    os.makedirs(model_dir, exist_ok=True)

    print(f"Loading training data from {train_dir}")
    X_train, y_train = load_channel(train_dir)
    X_val, y_val = load_channel(val_dir)
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")

    model = GradientBoostingClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        learning_rate=args.learning_rate,
        random_state=42,
    )
    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]
    val_preds = model.predict(X_val)
    auc = roc_auc_score(y_val, val_proba)
    f1 = f1_score(y_val, val_preds)
    print(f"Validation AUC: {auc:.4f}  F1: {f1:.4f}")

    # Save model artifact
    with open(os.path.join(model_dir, "model.pkl"), "wb") as f:
        pickle.dump(model, f)

    # Save metrics
    metrics = {"val_auc": round(auc, 4), "val_f1": round(f1, 4)}
    with open(os.path.join(model_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"Model saved to {model_dir}")
'''

# Write to a temp file for local execution
script_path = Path("src/train_sagemaker.py")
script_path.parent.mkdir(exist_ok=True)
script_path.write_text(training_script.strip())
print(f"Training script saved to {script_path}")

---
## 4. Local Training Baseline

Run the SageMaker-compatible script locally by setting environment variables.

In [ ]:
import subprocess

env = os.environ.copy()
env["SM_CHANNEL_TRAIN"] = str(Path("data/channels/train").resolve())
env["SM_CHANNEL_VALIDATION"] = str(Path("data/channels/validation").resolve())
env["SM_MODEL_DIR"] = str(Path("outputs/model_baseline").resolve())

start = time.time()
result = subprocess.run(
    [sys.executable, "src/train_sagemaker.py",
     "--n-estimators", "100", "--max-depth", "5", "--learning-rate", "0.1"],
    env=env, capture_output=True, text=True
)
elapsed = time.time() - start

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"\nBaseline training took {elapsed:.1f}s")

In [ ]:
# Load and inspect baseline model
baseline_dir = Path("outputs/model_baseline")
with open(baseline_dir / "model.pkl", "rb") as f:
    baseline_model = pickle.load(f)
with open(baseline_dir / "metrics.json") as f:
    baseline_metrics = json.load(f)

print("Baseline metrics:", baseline_metrics)

# Evaluate on test set
test_proba = baseline_model.predict_proba(X_test)[:, 1]
test_preds = baseline_model.predict(X_test)
print(f"\nTest AUC:  {roc_auc_score(y_test, test_proba):.4f}")
print(f"Test F1:   {f1_score(y_test, test_preds):.4f}")
print("\n", classification_report(y_test, test_preds))

---
## 5. Hyperparameter Search

Local equivalent of SageMaker HPO — use `RandomizedSearchCV` to search across multiple model families.

In [ ]:
# TODO: Hyperparameter search across multiple models
#
# Models to compare:
#   1. LogisticRegression  — C, penalty
#   2. RandomForestClassifier — n_estimators, max_depth, min_samples_split
#   3. GradientBoostingClassifier — n_estimators, max_depth, learning_rate
#   4. XGBClassifier (if xgboost installed) — same as GBM params
#
# Approach:
#   - Use RandomizedSearchCV with scoring='roc_auc', cv=3
#   - n_iter=20 per model (or less for faster iteration)
#   - Store best estimator and score for each model family
#
# Example skeleton:
# results = {}
# for name, model, param_dist in model_configs:
#     search = RandomizedSearchCV(model, param_dist, n_iter=20,
#                                  scoring='roc_auc', cv=3, random_state=42, n_jobs=-1)
#     search.fit(X_train, y_train)
#     results[name] = {
#         'best_estimator': search.best_estimator_,
#         'best_score': search.best_score_,
#         'best_params': search.best_params_,
#     }
#     print(f"{name}: CV AUC = {search.best_score_:.4f}")

print("TODO: Implement hyperparameter search")

---
## 6. Multi-Model Comparison

Compare all tuned models on the validation set.

In [ ]:
# TODO: Model comparison on validation set
#
# For each model in results:
#   - Predict on X_val
#   - Compute AUC, F1, Precision, Recall
#   - Store in a comparison DataFrame
#
# comparison_df = pd.DataFrame(comparison_rows)
# comparison_df = comparison_df.sort_values('val_auc', ascending=False)
# display(comparison_df)

print("TODO: Implement model comparison table")

In [ ]:
# TODO: Bar chart comparing model performance
#
# fig, ax = plt.subplots(figsize=(10, 5))
# comparison_df.set_index('model')[['val_auc', 'val_f1']].plot.bar(ax=ax)
# ax.set_title('Model Comparison — Validation Metrics')
# ax.set_ylabel('Score')
# ax.set_ylim(0.5, 1.0)
# plt.xticks(rotation=0)
# plt.tight_layout()
# plt.savefig('outputs/model_comparison.png', dpi=150)
# plt.show()

print("TODO: Implement comparison bar chart")

---
## 7. Model Artifact Packaging

Save the best model as a deployment-ready artifact with metadata.

In [ ]:
# TODO: Package best model as deployment-ready artifact
#
# Steps:
# 1. Select best model from comparison (highest val_auc)
# 2. Evaluate on test set for final metrics
# 3. Save artifact bundle:
#    outputs/best_model/
#      model.pkl          — serialized model
#      metadata.json      — hyperparams, metrics, timestamp
#      feature_names.json — ordered feature list
#
# artifact_dir = Path('outputs/best_model')
# artifact_dir.mkdir(parents=True, exist_ok=True)
#
# best_name = comparison_df.iloc[0]['model']
# best_model = results[best_name]['best_estimator']
#
# # Final test evaluation
# test_proba = best_model.predict_proba(X_test)[:, 1]
# test_preds = best_model.predict(X_test)
#
# metadata = {
#     'model_name': best_name,
#     'hyperparameters': results[best_name]['best_params'],
#     'cv_auc': results[best_name]['best_score'],
#     'test_auc': roc_auc_score(y_test, test_proba),
#     'test_f1': f1_score(y_test, test_preds),
#     'n_features': len(feature_cols),
#     'n_training_rows': X_train.shape[0],
#     'timestamp': pd.Timestamp.now().isoformat(),
# }
#
# with open(artifact_dir / 'model.pkl', 'wb') as f:
#     pickle.dump(best_model, f)
# with open(artifact_dir / 'metadata.json', 'w') as f:
#     json.dump(metadata, f, indent=2)
# with open(artifact_dir / 'feature_names.json', 'w') as f:
#     json.dump(feature_cols, f)
#
# print(f"Artifact saved to {artifact_dir}")
# print(json.dumps(metadata, indent=2))

print("TODO: Package best model artifact")

---
## 8. Optional: SageMaker Deployment

If you have AWS credentials and a SageMaker role, here is how you would deploy the training to real SageMaker infrastructure.

```python
# import sagemaker
# from sagemaker.sklearn import SKLearn
#
# session = sagemaker.Session()
# role = 'arn:aws:iam::ACCOUNT_ID:role/SageMakerRole'
# bucket = session.default_bucket()
#
# # Upload data to S3
# train_s3 = session.upload_data('data/channels/train', key_prefix='p2-distributed/train')
# val_s3 = session.upload_data('data/channels/validation', key_prefix='p2-distributed/validation')
#
# # Create SKLearn Estimator
# estimator = SKLearn(
#     entry_point='src/train_sagemaker.py',
#     role=role,
#     instance_count=1,
#     instance_type='ml.m5.xlarge',
#     framework_version='1.2-1',
#     hyperparameters={
#         'n-estimators': 200,
#         'max-depth': 6,
#         'learning-rate': 0.05,
#     },
# )
#
# # Launch training job
# estimator.fit({'train': train_s3, 'validation': val_s3})
#
# # Deploy as real-time endpoint
# predictor = estimator.deploy(
#     initial_instance_count=1,
#     instance_type='ml.m5.large',
# )
```

**Key points:**
- The same `train_sagemaker.py` script runs both locally and on SageMaker
- SageMaker sets `SM_CHANNEL_TRAIN`, `SM_CHANNEL_VALIDATION`, `SM_MODEL_DIR` automatically
- For hyperparameter tuning at scale, use `sagemaker.tuner.HyperparameterTuner`
- Distributed training across multiple instances uses `SM_HOSTS` / `SM_CURRENT_HOST` env vars

---
## 9. What I Built / What I Learned

### Summary
- TODO: Summarize the end-to-end workflow

### Key Learnings
- TODO: What did you learn about SageMaker Script Mode?
- TODO: How does local-first development speed up iteration?
- TODO: What was the best model and why?
- TODO: What would you do differently with real SageMaker infrastructure?

### Results
- TODO: Best model name and test AUC
- TODO: Training time comparison
- TODO: Most important features for churn prediction